# Exploration des événements d'Auvergne-Rhône-Alpes

Ce notebook permet d'analyser les données d'événements brutes récupérées depuis l'API OpenAgenda et enregistrées dans `data/events_raw.json`.

In [18]:
# Bibliothèques standard Python
import json
import os
from datetime import datetime, timezone, timedelta
import pandas as pd

# Bibliothèques externes
import requests
from dotenv import load_dotenv


## 1. Chargement des données

In [19]:
# Charger le fichier JSON
with open('../data/events_raw.json', 'r', encoding='utf-8') as f:
    events_data = json.load(f)

print(f"Nombre d'événements chargés : {len(events_data)}")

Nombre d'événements chargés : 6013


## 2. Transformation en DataFrame Pandas

In [21]:
# Aplatir les données pour créer un DataFrame propre
df = pd.DataFrame(events_data)
# Voir toutes les colonnes disponibles
print(df.columns.tolist())



['image', 'featured', 'attendanceMode', 'keywords', 'dateRange', 'timezone', 'imageCredits', 'originAgenda', 'description', 'title', 'onlineAccessLink', 'uid', 'lastTiming', 'types-de-public', 'firstTiming', 'theme', 'location', 'cibles', 'slug', 'status', 'nextTiming', 'parent_agenda_uid', 'organisateur', 'thematique', 'evenement-europe', 'particularite', 'portee-de-levenement', 'si-joli-mois-de-leurope-nombre-de-participants-attendus-en-presentiel', 'type-de-public', 'actions', 'conditions-de-participation', 'theme-jep', 'types-devenement', 'jaccepte-que-limage-puisse-etre-librement-utilisee-a-la-condition', 'specificites', 'domaine-artistique', 'publics-vises', 'type-devenement', 'tag-group', 'category-group', 'element-patrimoine-culturel-immateriel', 'la-french-tech-organise-ou-coorganise-levenement', 'thematiques', 'categories', 'numero-de-lorganisateur', 'le-sejour-se-deroule', 'qualifications', 'actions-retenues-pour-favoriser-la-participation-des-enfants-et-des-jeunes', 'vacanc

In [22]:
print(json.dumps(events_data[0], indent=2, ensure_ascii=False))


{
  "image": {
    "filename": "05ec99b07dcc48eca79924b4d824db9d.base.image.jpg",
    "size": {
      "width": 700,
      "height": 420
    },
    "variants": [
      {
        "filename": "05ec99b07dcc48eca79924b4d824db9d.full.image.jpg",
        "size": {
          "width": 2500,
          "height": 1500
        },
        "type": "full"
      },
      {
        "filename": "05ec99b07dcc48eca79924b4d824db9d.thumb.image.jpg",
        "size": {
          "width": 200,
          "height": 200
        },
        "type": "thumbnail"
      }
    ],
    "base": "https://cdn.openagenda.com/main/"
  },
  "featured": false,
  "attendanceMode": 1,
  "keywords": {
    "fr": [
      "hydrologie régénérative ; viticulture ; vignoble ; gestion de l'eau ; adaptation climatique ; stress hydrique"
    ]
  },
  "dateRange": {
    "ar": "الخميس ٢ يوليو, 08:00",
    "de": "Donnerstag 2 Juli, 08:00",
    "en": "Thursday 2 July, 08:00",
    "it": "Giovedì 2 luglio, 08:00",
    "fr": "Jeudi 2 juillet, 08h00

In [ ]:
df['departement'].unique()

<StringArray>
['Inconnu']
Length: 1, dtype: str

In [23]:
# Colonnes qu'on garde pour le RAG
colonnes_utiles = [
    'uid',
    'title',
    'description',
    'location',
    'firstTiming',
    'lastTiming',
    'slug',
    'keywords',
    'theme',
    'cibles',
    'public',
    'types-de-public',
    'conditions-de-participation',
    'attendanceMode',
    'organisateur'
]

# Garder seulement les colonnes qui existent dans le DataFrame
colonnes_presentes = [c for c in colonnes_utiles if c in df.columns]

# Créer le DataFrame nettoyé
df_clean = df[colonnes_presentes].copy()

print(f"Colonnes gardées : {colonnes_presentes}")
print(f"Nombre d'événements : {len(df_clean)}")
df_clean.head()

Colonnes gardées : ['uid', 'title', 'description', 'location', 'firstTiming', 'lastTiming', 'slug', 'keywords', 'theme', 'cibles', 'public', 'types-de-public', 'conditions-de-participation', 'attendanceMode', 'organisateur']
Nombre d'événements : 6013


,uid,title,description,location,firstTiming,lastTiming,slug,keywords,theme,cibles,public,types-de-public,conditions-de-participation,attendanceMode,organisateur
0,95768697,{'fr': 'Journée Technique Hydrologie regénérat...,{'fr': 'demi-journée technique gratuite et ouv...,"{'address': '2, place de la salle des fêtes 07...","{'end': '2026-07-02T12:00:00.000+02:00', 'begi...","{'end': '2026-07-02T12:00:00.000+02:00', 'begi...",journee-technique-hydrologie-regenerative-en-f...,{'fr': ['hydrologie régénérative ; viticulture...,[169],[138],NaN,[137],NaN,1,NaN
1,89578114,{'fr': 'Webinaire – La réforme de la facturati...,{'fr': 'La CA07 organise un webinaire pour com...,"{'address': '07000', 'city': 'Veyras', 'latitu...","{'end': '2026-07-07T15:00:00.000+02:00', 'begi...","{'end': '2026-07-07T15:00:00.000+02:00', 'begi...",webinaire-la-reforme-de-la-facturation-electro...,"{'fr': ['facturation électronique', 'entrepris...",[155],[138],NaN,[137],NaN,1,NaN
2,65182490,{'fr': 'Webinaire Castanea Sativa'},{'fr': 'La Chambre d'agriculture de l'Ardèch p...,"{'address': '07000', 'city': 'Veyras', 'latitu...","{'end': '2026-07-08T05:00:00.000+02:00', 'begi...","{'end': '2026-07-08T05:00:00.000+02:00', 'begi...",webinaire-castanea-sativa,{'fr': ['châtaignier ; castanéiculture ; Casta...,[167],[138],NaN,[137],NaN,2,NaN
3,36521668,{'fr': 'Estive en fête !'},{'fr': 'Randonnée pastorale festive sur les ha...,"{'address': 'la croix de Bauzon, Ardèche', 'ci...","{'end': '2026-07-11T16:00:00.000+02:00', 'begi...","{'end': '2026-07-11T16:00:00.000+02:00', 'begi...",estive-en-fete,"{'fr': ['pastoralisme', 'Tanargue', 'randonnée...",[176],[146],NaN,[136],NaN,1,NaN
4,36284787,{'fr': 'Facturation électronique'},{'fr': 'Réunion d’information interconsulaire ...,"{'address': 'coucouron', 'city': 'Coucouron', ...","{'end': '2026-06-01T17:00:00.000+02:00', 'begi...","{'end': '2026-06-01T17:00:00.000+02:00', 'begi...",facturation-electronique-4458635,"{'fr': ['facturation électronique', 'Ardèche']}",[156],[138],NaN,[137],NaN,1,NaN


In [24]:
# Vérifier le % de valeurs manquantes par colonne
nan_stats = df_clean.isnull().sum() / len(df_clean) * 100
print("% de valeurs manquantes par colonne :")
print(nan_stats.round(1))

% de valeurs manquantes par colonne :
uid                             0.0
title                           0.0
description                     0.0
location                        0.5
firstTiming                     0.0
lastTiming                      0.0
slug                            0.0
keywords                        0.0
theme                          88.6
cibles                         97.0
public                         99.6
types-de-public                97.0
conditions-de-participation    59.0
attendanceMode                  0.0
organisateur                   95.4
dtype: float64


In [26]:
# Voir le contenu brut des keywords
print(df_clean['keywords'].iloc[0])
print(type(df_clean['keywords'].iloc[0]))

{'fr': ["hydrologie régénérative ; viticulture ; vignoble ; gestion de l'eau ; adaptation climatique ; stress hydrique"]}
<class 'dict'>


In [27]:
# Extraire les keywords en français
df_clean['keywords_fr'] = df_clean['keywords'].apply(
    lambda x: x.get('fr', '') if isinstance(x, dict) else ''
)

# Vérifier
print(df_clean['keywords_fr'].iloc[0])

["hydrologie régénérative ; viticulture ; vignoble ; gestion de l'eau ; adaptation climatique ; stress hydrique"]


In [28]:
# Extraire title et description en français
df_clean['title_fr'] = df_clean['title'].apply(
    lambda x: x.get('fr', '') if isinstance(x, dict) else str(x)
)

df_clean['description_fr'] = df_clean['description'].apply(
    lambda x: x.get('fr', '') if isinstance(x, dict) else str(x)
)

# Vérifier
print(df_clean['title_fr'].iloc[0])
print("---")
print(df_clean['description_fr'].iloc[0])

Journée Technique Hydrologie regénérative en forte pente
---
demi-journée technique gratuite et ouverte à tous les professionnels.


In [29]:
# DataFrame final avec colonnes lisibles
df_final = df_clean[[
    'uid', 'slug', 'title_fr', 'description_fr', 
    'keywords_fr', 'location', 'firstTiming', 
    'lastTiming', 'attendanceMode'
]].copy()

print(f"Shape final : {df_final.shape}")
df_final.head(2)

Shape final : (6013, 9)


,uid,slug,title_fr,description_fr,keywords_fr,location,firstTiming,lastTiming,attendanceMode
0,95768697,journee-technique-hydrologie-regenerative-en-f...,Journée Technique Hydrologie regénérative en f...,demi-journée technique gratuite et ouverte à t...,[hydrologie régénérative ; viticulture ; vigno...,"{'address': '2, place de la salle des fêtes 07...","{'end': '2026-07-02T12:00:00.000+02:00', 'begi...","{'end': '2026-07-02T12:00:00.000+02:00', 'begi...",1
1,89578114,webinaire-la-reforme-de-la-facturation-electro...,Webinaire – La réforme de la facturation élect...,La CA07 organise un webinaire pour comprendre ...,"[facturation électronique, entreprise agricole...","{'address': '07000', 'city': 'Veyras', 'latitu...","{'end': '2026-07-07T15:00:00.000+02:00', 'begi...","{'end': '2026-07-07T15:00:00.000+02:00', 'begi...",1


In [30]:
# Vérifier les NaN sur le DataFrame final
print(df_final.isnull().sum())

uid                0
slug               0
title_fr           0
description_fr     0
keywords_fr        0
location          29
firstTiming        0
lastTiming         0
attendanceMode     0
dtype: int64


In [31]:
# Supprimer les 29 lignes sans location
df_final = df_final.dropna(subset=['location'])
print(f"Événements restants : {len(df_final)}")

Événements restants : 5984


In [32]:
# Sauvegarder le fichier propre
df_final.to_json(
    '../data/events_clean.json', 
    orient='records', 
    force_ascii=False, 
    indent=2
)
print("✅ Sauvegardé dans data/events_clean.json")

✅ Sauvegardé dans data/events_clean.json


## 3. Analyse et visualisations

### A. Répartition des événements par département

### B. Top 10 des villes accueillant le plus d'événements